# Export 5-Fold Cross Validation Data ke File Excel
Notebook ini digunakan untuk mengeksport data dari setiap fold (Training & Testing) ke file Excel terpisah.

## 1. Import Libraries

In [6]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Dataset

In [7]:
# Load dataset
df = pd.read_csv('PhiUSIIL_Phishing_URL_59_Features.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(f"  Non-Malicious (0): {(df['label']==0).sum():,}")
print(f"  Malicious (1):     {(df['label']==1).sum():,}")
print(f"  Total:             {len(df):,}")

Dataset shape: (235795, 59)

Label distribution:
  Non-Malicious (0): 100,945
  Malicious (1):     134,850
  Total:             235,795


## 3. Data Preprocessing

In [8]:
# Kolom non-numerik yang harus di-drop
non_numeric_cols = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']

# Drop kolom non-numerik
df_numeric = df.drop(columns=non_numeric_cols, errors='ignore')

# Pisahkan fitur dan target
X = df_numeric.drop(columns=['label'])
y = df_numeric['label']

# Handle missing values
X = X.fillna(X.median())

# Pastikan semua kolom numerik
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.median())

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (235795, 53)
Target shape: (235795,)


## 4. Konfigurasi Fold
**Anda bisa mengubah jumlah fold di sini!**

In [9]:
# ============================================================
# KONFIGURASI - UBAH NILAI INI SESUAI KEBUTUHAN
# ============================================================
N_SPLITS = 5  # Jumlah Fold (ubah sesuai kebutuhan: 3, 5, 10, dll)
RANDOM_STATE = 42  # Random seed untuk reproducibility
OUTPUT_FOLDER = 'Fold_Data'  # Folder output

print(f"Konfigurasi:")
print(f"  - Jumlah Fold: {N_SPLITS}")
print(f"  - Random State: {RANDOM_STATE}")
print(f"  - Output Folder: {OUTPUT_FOLDER}/")

Konfigurasi:
  - Jumlah Fold: 5
  - Random State: 42
  - Output Folder: Fold_Data/


## 5. Export 5 Fold ke File CSV Terpisah
Setiap fold = 20% data (~47,159 samples dengan distribusi NM ≈ 20,189 dan M ≈ 26,970)

In [10]:
# ============================================================
# EXPORT 5 FOLD KE FILE CSV TERPISAH
# Setiap fold = 20% data (~47,159 samples)
# ============================================================

# Buat folder untuk menyimpan file fold
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Setup Stratified K-Fold Cross Validation
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Gabungkan X dan y untuk export
df_full = X.copy()
df_full['label'] = y.values

# Hitung ukuran per fold
samples_per_fold = len(df_full) // N_SPLITS
nm_per_fold = (y == 0).sum() // N_SPLITS
m_per_fold = (y == 1).sum() // N_SPLITS

print("="*70)
print(f"EXPORT {N_SPLITS} FOLD KE FILE CSV TERPISAH")
print("="*70)
print(f"\n📊 TOTAL DATASET: {len(df_full):,} samples")
print(f"   Non-Malicious (0): {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.2f}%)")
print(f"   Malicious (1):     {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.2f}%)")

print(f"\n📐 PEMBAGIAN PER FOLD (masing-masing 20%):")
print(f"   Setiap 1 Fold ≈ {samples_per_fold:,} samples")
print(f"     - NM per fold ≈ {nm_per_fold:,}")
print(f"     - M per fold  ≈ {m_per_fold:,}")

print(f"\n🔄 KONSEP PENGGUNAAN NANTI:")
print(f"   • TRAINING = 4 Fold (pilih 4 file CSV)")
print(f"   • TESTING  = 1 Fold (pilih 1 file CSV)")
print(f"   • Total ada 5 kombinasi iterasi")

# Loop setiap fold dan export ke CSV terpisah
for fold_num, (_, fold_idx) in enumerate(skf.split(X, y), 1):
    
    # Data untuk fold ini
    df_fold = df_full.iloc[fold_idx].copy()
    
    # Statistik
    fold_nm = (df_fold['label'] == 0).sum()
    fold_m = (df_fold['label'] == 1).sum()
    
    print(f"\n{'='*70}")
    print(f"FOLD {fold_num}")
    print(f"{'='*70}")
    print(f"  📄 Total samples: {len(df_fold):,} ({len(df_fold)/len(df_full)*100:.1f}%)")
    print(f"      - Non-Malicious (0): {fold_nm:,} ({fold_nm/len(df_fold)*100:.2f}%)")
    print(f"      - Malicious (1):     {fold_m:,} ({fold_m/len(df_fold)*100:.2f}%)")
    
    # Export ke CSV
    csv_path = f'{OUTPUT_FOLDER}/Fold_{fold_num}.csv'
    df_fold.to_csv(csv_path, index=False)
    
    print(f"  ✅ Exported to: {csv_path}")

print("\n" + "="*70)
print("✅ SEMUA 5 FILE CSV BERHASIL DI-EXPORT!")
print("="*70)
print(f"\n📁 Folder: {OUTPUT_FOLDER}/")
print("📄 File yang dihasilkan:")
for i in range(1, N_SPLITS + 1):
    print(f"   {i}. Fold_{i}.csv")

EXPORT 5 FOLD KE FILE CSV TERPISAH

📊 TOTAL DATASET: 235,795 samples
   Non-Malicious (0): 100,945 (42.81%)
   Malicious (1):     134,850 (57.19%)

📐 PEMBAGIAN PER FOLD (masing-masing 20%):
   Setiap 1 Fold ≈ 47,159 samples
     - NM per fold ≈ 20,189
     - M per fold  ≈ 26,970

🔄 KONSEP PENGGUNAAN NANTI:
   • TRAINING = 4 Fold (pilih 4 file CSV)
   • TESTING  = 1 Fold (pilih 1 file CSV)
   • Total ada 5 kombinasi iterasi

FOLD 1
  📄 Total samples: 47,159 (20.0%)
      - Non-Malicious (0): 20,189 (42.81%)
      - Malicious (1):     26,970 (57.19%)
  ✅ Exported to: Fold_Data/Fold_1.csv

FOLD 2
  📄 Total samples: 47,159 (20.0%)
      - Non-Malicious (0): 20,189 (42.81%)
      - Malicious (1):     26,970 (57.19%)
  ✅ Exported to: Fold_Data/Fold_2.csv

FOLD 3
  📄 Total samples: 47,159 (20.0%)
      - Non-Malicious (0): 20,189 (42.81%)
      - Malicious (1):     26,970 (57.19%)
  ✅ Exported to: Fold_Data/Fold_3.csv

FOLD 4
  📄 Total samples: 47,159 (20.0%)
      - Non-Malicious (0): 20,189

## 6. Daftar File yang Dihasilkan

In [11]:
# Tampilkan daftar file yang dihasilkan
print(f"\n📁 Files created in '{OUTPUT_FOLDER}/' folder:")
print("="*60)

files = os.listdir(OUTPUT_FOLDER)
csv_files = [f for f in files if f.endswith('.csv')]

for f in sorted(csv_files):
    file_path = os.path.join(OUTPUT_FOLDER, f)
    file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
    
    # Baca file untuk statistik
    df_temp = pd.read_csv(file_path)
    nm_count = (df_temp['label'] == 0).sum()
    m_count = (df_temp['label'] == 1).sum()
    
    print(f"   • {f}")
    print(f"     Size: {file_size:.2f} MB | Rows: {len(df_temp):,}")
    print(f"     NM: {nm_count:,} | M: {m_count:,}")

print(f"\n📐 CARA PENGGUNAAN:")
print("="*60)
print("   Contoh Iterasi 1: Training = Fold 1+2+3+4, Testing = Fold 5")
print("   Contoh Iterasi 2: Training = Fold 1+2+3+5, Testing = Fold 4")
print("   Contoh Iterasi 3: Training = Fold 1+2+4+5, Testing = Fold 3")
print("   Contoh Iterasi 4: Training = Fold 1+3+4+5, Testing = Fold 2")
print("   Contoh Iterasi 5: Training = Fold 2+3+4+5, Testing = Fold 1")


📁 Files created in 'Fold_Data/' folder:
   • Fold_1.csv
     Size: 7.90 MB | Rows: 47,159
     NM: 20,189 | M: 26,970
   • Fold_2.csv
     Size: 7.90 MB | Rows: 47,159
     NM: 20,189 | M: 26,970
   • Fold_3.csv
     Size: 7.90 MB | Rows: 47,159
     NM: 20,189 | M: 26,970
   • Fold_4.csv
     Size: 7.89 MB | Rows: 47,159
     NM: 20,189 | M: 26,970
   • Fold_5.csv
     Size: 7.90 MB | Rows: 47,159
     NM: 20,189 | M: 26,970

📐 CARA PENGGUNAAN:
   Contoh Iterasi 1: Training = Fold 1+2+3+4, Testing = Fold 5
   Contoh Iterasi 2: Training = Fold 1+2+3+5, Testing = Fold 4
   Contoh Iterasi 3: Training = Fold 1+2+4+5, Testing = Fold 3
   Contoh Iterasi 4: Training = Fold 1+3+4+5, Testing = Fold 2
   Contoh Iterasi 5: Training = Fold 2+3+4+5, Testing = Fold 1


## 7. (Opsional) Preview Data dari Salah Satu Fold

In [12]:
# Preview data dari salah satu fold
fold_preview = 1  # Ubah nomor fold yang ingin di-preview (1-5)

csv_file = f'{OUTPUT_FOLDER}/Fold_{fold_preview}.csv'

print(f"📄 Preview data dari: {csv_file}")
print("="*60)

# Read CSV
df_preview = pd.read_csv(csv_file)

print(f"\n📊 Statistik Fold {fold_preview}:")
print(f"   Total rows: {len(df_preview):,}")
print(f"   Total columns: {len(df_preview.columns)}")
print(f"   Non-Malicious (0): {(df_preview['label']==0).sum():,}")
print(f"   Malicious (1): {(df_preview['label']==1).sum():,}")

print(f"\n📋 Columns:")
print(df_preview.columns.tolist())

print(f"\n📝 First 10 rows:")
print(df_preview.head(10))

📄 Preview data dari: Fold_Data/Fold_1.csv

📊 Statistik Fold 1:
   Total rows: 47,159
   Total columns: 54
   Non-Malicious (0): 20,189
   Malicious (1): 26,970

📋 Columns:
['URLLength', 'DomainLength', 'IsDomainIP', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRe